# Week 6: Getting Earth Observation data into Python

**GEOG0111 Scientific Computing**
MSc Environmental Modelling · MSc Remote Sensing and Environmental Mapping · UCL Geography

From this week on we work with **real Earth Observation (EO) data**. You already have the building blocks: functions (Week 3), files and URLs (Week 4), and plots and images as arrays (Week 5). Now we connect them to the three big families of geospatial data, and to the three main ways of getting them:

| | Data | Today's example | Python tools |
|---|---|---|---|
| 1 | **Vector**: points, lines, polygons with attributes | London borough boundaries | `geopandas` |
| 2 | **Raster** read from the cloud: files you read directly from a server | Sentinel-2 over London via a **STAC** catalogue | `pystac-client`, `rioxarray` |
| 3 | **Raster** processed in the cloud: send the computation, not the data | Sentinel-2, Landsat temperature and forest loss via **Google Earth Engine** | `earthengine-api`, `geemap` |

All examples lead straight into your **Part B project** (section 6): London's urban heat island and Amazon deforestation.

**How to use this notebook:** it is a worked walk-through. Run the cells in order, read the explanations, and stop at the **Questions**. Think about your answer first, then click **Show answer**. Change values (dates, areas, thresholds) and rerun cells to see what happens: that is the best way to learn these tools.

## What you will learn

- the difference between downloading data, reading cloud-optimised files, and processing in Earth Engine, and when to use each;
- loading vector data with `geopandas`, checking its **CRS**, reprojecting it, calculating areas and saving it;
- searching a **STAC** catalogue for Sentinel-2 scenes, reading only the part of a cloud-optimised GeoTIFF (COG) you need, and **mosaicking** two tiles;
- converting stored digital numbers to **surface reflectance**, and checking that the result makes sense;
- masking clouds, clipping a raster to a polygon and calculating **zonal statistics** (mean NDVI per borough);
- building a **cloud-masked composite** in Google Earth Engine, calculating NDVI and **land surface temperature**, summarising by borough, and exporting a GeoTIFF;
- finding datasets in the official and community Earth Engine catalogues;
- what Part B requires, and how today's pieces fit into it.

### Before you start

- **Google Earth Engine account** (needed for section 4): follow the steps in section 4.1, **before the session** if you can.
- Every section that needs the internet or Earth Engine **checks first and skips cleanly** if something is unavailable, so the rest still runs.
- The module's AI rules apply. In EO work, AI tools often invent **dataset IDs and band names that do not exist**: always check them against the catalogue page (section 5).

In [ ]:
# Geospatial packages that Colab does not always have (about 1 minute)
%pip install -q geopandas rioxarray pystac-client earthengine-api geemap

In [ ]:
from pathlib import Path
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rioxarray as rxr
import xarray as xr
from shapely.geometry import shape, box
from shapely.ops import unary_union
from rasterio.enums import Resampling

DATA_DIR = Path('data')
FIG_DIR = DATA_DIR / 'figures'
for d in [DATA_DIR / 'raw', DATA_DIR / 'processed', FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Study area used all week: Greater London (lon/lat bounding box from Week 2)
LONDON_BBOX = (-0.51, 51.28, 0.33, 51.69)    # (min_lon, min_lat, max_lon, max_lat)

print('geopandas', gpd.__version__, '| rioxarray', rxr.__version__)

---
# 1. Three ways to get EO data

| Approach | How it works | Good for | Watch out for |
|---|---|---|---|
| **Download** files (portal or URL) | Click and download, or `requests.get(url)`, then read the local copy | Small datasets, station data (Week 4), boundaries | Big files are slow; manual downloads are hard to reproduce |
| **Read from the cloud** (COG + STAC) | A catalogue (**STAC**) tells you which files exist; you read only the pixels you need from **cloud-optimised GeoTIFFs** | Single scenes, a few dates, full control of the processing | You handle reprojection, scaling and masking yourself |
| **Process in the cloud** (Google Earth Engine) | You send instructions; Google's servers run them over petabytes of data; you get back a map, numbers or a small export | Long time series, large areas, composites, many datasets | Needs an account; a different (server-side) programming model; export limits |

Professional EO work combines all three. A typical Part B project downloads a station record and a boundary (approach 1), and processes satellite imagery in Earth Engine (approach 3), exporting small results to analyse and plot in Python.

### Key vocabulary

- **CRS (coordinate reference system):** how coordinates relate to places on Earth. **EPSG:4326** = longitude/latitude in degrees. **EPSG:27700** = British National Grid in metres. **EPSG:326xx** = UTM zones in metres (Sentinel-2 tiles over London are in UTM zones 30N and 31N).
- **Resolution:** *spatial* (pixel size, e.g. 10 m), *temporal* (revisit, e.g. 5 days), *spectral* (number and width of bands), *radiometric* (bit depth).
- **Processing level:** e.g. Sentinel-2 **L1C** = top-of-atmosphere reflectance; **L2A** = bottom-of-atmosphere (**surface**) reflectance, which is what you normally want.

---
# 2. Vector data with `geopandas`: London's boroughs

A `GeoDataFrame` is a pandas DataFrame (Week 4) with an extra **`geometry`** column and a **CRS**. Everything you know about selecting, filtering and grouping still works.

We get the boundaries from **geoBoundaries**, an open database of administrative boundaries with a simple **API**. We ask the API about the UK's "Counties and Unitary Authorities" (Office for National Statistics data), and it replies with JSON metadata that includes the download link: *API → metadata → file URL → data*.

In [ ]:
def get_geoboundaries(iso3, level, out_path, simplified=True):
    """Download a geoBoundaries layer (via its API) and return it as a GeoDataFrame.

    Parameters
    ----------
    iso3 : str
        Country code, e.g. 'GBR'.
    level : str
        Administrative level, e.g. 'ADM3'.
    out_path : str or Path
        Local cache file (GeoJSON).
    simplified : bool, optional
        Use the simplified (smaller, faster) geometries (default True).

    Returns
    -------
    geopandas.GeoDataFrame
    """
    out_path = Path(out_path)
    if not out_path.exists():
        meta = requests.get(f'https://www.geoboundaries.org/api/current/gbOpen/{iso3}/{level}/', timeout=60).json()
        print('Dataset:', meta.get('boundaryCanonical'), '| source:', meta.get('boundarySource'),
              '| year:', meta.get('boundaryYearRepresented'))
        url = meta['simplifiedGeometryGeoJSON'] if simplified else meta['gjDownloadURL']
        r = requests.get(url, timeout=120)        # github.com/.../raw/... redirects to the real file
        r.raise_for_status()
        out_path.write_bytes(r.content)
    return gpd.read_file(out_path)


try:
    uk = get_geoboundaries('GBR', 'ADM3', DATA_DIR / 'raw' / 'gbr_adm3_simplified.geojson')
    BOUNDARIES_OK = True
    print(len(uk), 'units | CRS:', uk.crs)
    display(uk.head(3))
except Exception as e:
    BOUNDARIES_OK = False
    print(f'Could not get boundaries ({type(e).__name__}: {e}). Vector cells below will be skipped.')

The UK file contains 216 counties and unitary authorities. Greater London consists of **32 boroughs plus the City of London**. We select them **by name** and then **check** that we found all 33, because real data rarely match your expectations first time.

In [ ]:
LONDON_BOROUGHS = [
    'City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden',
    'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey',
    'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea',
    'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridge',
    'Richmond upon Thames', 'Southwark', 'Sutton', 'Tower Hamlets', 'Waltham Forest',
    'Wandsworth', 'Westminster',
]

if BOUNDARIES_OK:
    boroughs = uk[uk['shapeName'].isin(LONDON_BOROUGHS)][['shapeName', 'geometry']].reset_index(drop=True)
    boroughs = boroughs.rename(columns={'shapeName': 'borough'})
    missing = set(LONDON_BOROUGHS) - set(boroughs['borough'])
    assert len(boroughs) == 33, f'expected 33 boroughs, found {len(boroughs)}; missing: {missing}'
    print('All 33 London boroughs found. CRS:', boroughs.crs)

## 2.1 CRS matters: degrees vs metres

The data are in **EPSG:4326** (longitude/latitude). To measure areas we **reproject** to the **British National Grid (EPSG:27700)**, whose units are metres. `dissolve()` merges all boroughs into **one Greater London polygon**, which is exactly the step the Part B urban heat island template asks for.

In [ ]:
if BOUNDARIES_OK:
    boroughs_bng = boroughs.to_crs(epsg=27700)
    london_bng = boroughs_bng.dissolve()            # one polygon for Greater London
    london = london_bng.to_crs(epsg=4326)

    boroughs_bng['area_km2'] = boroughs_bng.area / 1e6
    print('Greater London area:', round(london_bng.area.iloc[0] / 1e6), 'km²')

    ranked = boroughs_bng.sort_values('area_km2', ascending=False)[['borough', 'area_km2']].round(1)
    print('\nLargest:\n', ranked.head(3).to_string(index=False))
    print('\nSmallest:\n', ranked.tail(3).to_string(index=False))

In [ ]:
if BOUNDARIES_OK:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    boroughs_bng.plot(ax=axes[0], facecolor='whitesmoke', edgecolor='grey', linewidth=0.6)
    london_bng.boundary.plot(ax=axes[0], color='black', linewidth=1.5)
    for _, row in boroughs_bng.iterrows():
        p = row.geometry.representative_point()
        axes[0].annotate(row['borough'], (p.x, p.y), fontsize=5.5, ha='center')
    axes[0].set_title('London boroughs (British National Grid, EPSG:27700)')
    axes[0].set_xlabel('Easting (m)')
    axes[0].set_ylabel('Northing (m)')

    boroughs_bng.plot(ax=axes[1], column='area_km2', cmap='viridis', edgecolor='white', linewidth=0.4,
                      legend=True, legend_kwds={'label': 'Area (km²)', 'shrink': 0.7})
    axes[1].set_title('Borough area: a choropleth map')
    axes[1].set_axis_off()

    fig.tight_layout()
    plt.show()

    boroughs_bng.to_file(DATA_DIR / 'processed' / 'london_boroughs.gpkg')   # GeoPackage: one modern file
    print('saved london_boroughs.gpkg')

**Question:** What would `boroughs.area` (still in EPSG:4326) give you, and why can't you use it?

<details><summary><b>Show answer</b></summary>

It gives areas in **square degrees** (and geopandas warns you). A degree of latitude is about 111 km, but a degree of longitude at London's latitude is only about 69 km, and it shrinks further towards the poles. So a "square degree" is not a fixed area, and the numbers cannot be compared or converted to km². Always reproject to a projected CRS in metres (here EPSG:27700) before measuring areas or distances.

</details>

**Question:** Why save as a GeoPackage (`.gpkg`) rather than a shapefile?

<details><summary><b>Show answer</b></summary>

A shapefile is really 4+ separate files (`.shp`, `.shx`, `.dbf`, `.prj`...) that must stay together, limits column names to 10 characters, and has other old restrictions. A GeoPackage is one modern, open file that can hold several layers, keeps full column names and stores the CRS reliably.

</details>

---
# 3. Raster data from the cloud: Sentinel-2 via STAC

The Sentinel-2 archive is many petabytes in size, so you cannot download it. But it is stored openly on Amazon Web Services as **cloud-optimised GeoTIFFs (COGs)**: GeoTIFFs organised in internal tiles, with built-in low-resolution **overviews**, so a program can read **just a window**, or **just a coarse version**, over the internet.

To find the right files we use a **STAC** (SpatioTemporal Asset Catalog) API: a standard way to search EO archives by **area, date and properties** such as cloud cover. We use the free **Earth Search** catalogue (Element 84). The same code works with other STAC catalogues (Microsoft Planetary Computer, Digital Earth Africa, NASA).

## 3.1 Search the catalogue

In [ ]:
from pystac_client import Client

try:
    catalog = Client.open('https://earth-search.aws.element84.com/v1')
    search = catalog.search(
        collections=['sentinel-2-l2a'],              # Level-2A = surface reflectance
        bbox=LONDON_BBOX,
        datetime='2024-06-01/2024-08-31',
        query={'eo:cloud_cover': {'lt': 30}},         # scene-level cloud cover below 30 %
        max_items=200,
    )
    items = list(search.items())
    STAC_OK = len(items) > 0
    print(len(items), 'scenes found')
except Exception as e:
    STAC_OK = False
    print(f'STAC search failed ({type(e).__name__}: {e}). Section 3 will be skipped.')

Each **item** is one scene: one 110 × 110 km **tile** on one date. Each item has **assets**: one file per band (`red`, `nir`, `scl`, ...), plus a thumbnail and metadata.

London sits on the boundary between two **UTM zones** (30 and 31, divided at 0° longitude, close to Greenwich), so it is split across two Sentinel-2 tiles: **30UXC** (west) and **31UCT** (east). A single tile leaves part of London empty. So instead of choosing one scene, we choose the **date** whose scenes **together** cover all of London with the least cloud:

In [ ]:
def best_date(items, aoi):
    """Rank acquisition dates by how much of aoi their scenes cover together, then by cloud cover.

    Parameters
    ----------
    items : list of pystac.Item
    aoi : shapely geometry (lon/lat)

    Returns
    -------
    pandas.DataFrame
        One row per date: coverage (0-1), mean cloud cover (%), tiles and the items.
    """
    rows = []
    for date in sorted({it.datetime.date() for it in items}):
        day_items = [it for it in items if it.datetime.date() == date]
        footprint = unary_union([shape(it.geometry) for it in day_items])
        rows.append({'date': date,
                     'coverage': footprint.intersection(aoi).area / aoi.area,
                     'mean_cloud': np.mean([it.properties['eo:cloud_cover'] for it in day_items]),
                     'tiles': ', '.join(sorted(it.id.split('_')[1] for it in day_items)),
                     'items': day_items})
    df = pd.DataFrame(rows)
    df['coverage_rounded'] = df['coverage'].round(2)
    return df.sort_values(['coverage_rounded', 'mean_cloud'], ascending=[False, True]).drop(columns='coverage_rounded')


if STAC_OK:
    london_shape = london.geometry.iloc[0] if BOUNDARIES_OK else box(*LONDON_BBOX)
    dates = best_date(items, london_shape)
    display(dates.drop(columns='items').head(8).round(2))

    chosen = dates.iloc[0]
    day_items = chosen['items']
    print(f"Chosen date: {chosen['date']} | tiles: {chosen['tiles']} | covers {chosen['coverage']:.0%} of London")
    print('Assets of one item:', list(day_items[0].assets.keys())[:20])

## 3.2 Read only what we need

`rioxarray.open_rasterio()` opens a remote file **lazily**: nothing is downloaded until values are needed. `overview_level=0` asks for the **first** overview (overviews are counted from 0), which is 20 m instead of 10 m, i.e. 4× fewer pixels; `overview_level=1` would be 40 m, and `rio.clip_box()` restricts the read to London. Together, these mean we download a few MB instead of the full ~200 MB band.

Each tile is in its own UTM CRS, so we convert the London bounding box to that CRS before clipping.

In [ ]:
def read_band(item, asset, bounds_ll, overview_level=None):
    """Read one band of a STAC item, clipped to a lon/lat bounding box.

    Returns a 2-D xarray DataArray in the tile's own CRS, with the stored
    (unscaled) integer values.
    """
    href = item.assets[asset].href
    da = rxr.open_rasterio(href, overview_level=overview_level).squeeze('band', drop=True)
    xmin, ymin, xmax, ymax = gpd.GeoSeries([box(*bounds_ll)], crs=4326).to_crs(da.rio.crs).total_bounds
    return da.rio.clip_box(minx=xmin, miny=ymin, maxx=xmax, maxy=ymax)

## 3.3 From stored numbers to reflectance, and checking the metadata

The files store **integers** (`uint16`) to save space. **Surface reflectance** (0–1) is

$$\rho = DN \times \text{scale} + \text{offset}$$

and the item's metadata (`raster:bands`) gives the scale and offset. Since ESA processing baseline 04.00 (January 2022), raw Sentinel-2 L2A products need an **offset of −0.1** (−1000 in DN), as you saw in Week 3.

But look carefully at what Earth Search says about our scene:

In [ ]:
if STAC_OK:
    it = day_items[0]
    print('processing baseline:          ', it.properties.get('s2:processing_baseline'))
    print('earthsearch:boa_offset_applied:', it.properties.get('earthsearch:boa_offset_applied'))
    print('red raster:bands:             ', it.assets['red'].extra_fields.get('raster:bands'))

The metadata contradict each other. `raster:bands` lists an offset of −0.1, but `earthsearch:boa_offset_applied: True` says the offset has **already been applied** to the stored values, i.e. Earth Search has *harmonised* new scenes so that they behave like the pre-2022 ones. If we subtract 0.1 again, reflectance becomes far too low (urban red reflectance drops to about 0 or below), and NDVI becomes unrealistically high almost everywhere.

So our function respects the flag, and we then **check the result against what we know physically**: surface reflectance must be between 0 and 1, and typical red reflectance of a city in summer is a few per cent.

In [ ]:
def to_reflectance(da, item, asset):
    """Convert stored DN to surface reflectance, using the item's metadata.

    Uses raster:bands scale/offset/nodata, but skips the offset when
    Earth Search reports that it has already been applied to the data.
    """
    rb = item.assets[asset].extra_fields.get('raster:bands', [{}])[0]
    scale, offset, nodata = rb.get('scale', 1e-4), rb.get('offset', 0.0), rb.get('nodata', 0)
    if item.properties.get('earthsearch:boa_offset_applied'):
        offset = 0.0                                 # already applied: do not subtract twice
    return (da.where(da != nodata) * scale + offset).astype('float32')

**Question:** Why is "read the metadata, don't type the numbers" not quite enough here?

<details><summary><b>Show answer</b></summary>

Because metadata can be incomplete or inconsistent between fields (here `raster:bands` and `boa_offset_applied`), and conventions differ between data providers (ESA, Earth Search, Earth Engine). The professional habit is: read the documentation and metadata, apply the conversion, and then **sanity-check the numbers** against physics (reflectance between 0 and 1, plausible values for known surfaces such as water, vegetation and roofs). A quick histogram or a few summary statistics catches most of these errors.

</details>

## 3.4 Mosaic the two tiles and mask clouds

To combine tiles from **two different UTM zones**, we put them on one common grid: a 20 m grid in **British National Grid** covering London. Each tile is **reprojected** onto that grid, and the two are merged: where the first tile has no data, the second fills in.

(Elsewhere, prefer reprojecting the *vector* to the raster's CRS, because raster reprojection resamples pixel values. Combining zones is the exception where reprojecting the raster is unavoidable.)

We also mask clouds with the **Scene Classification Layer (SCL)**, a 20 m band in every L2A scene that labels each pixel (4 = vegetation, 5 = bare/built, 6 = water, 8–10 = cloud and cirrus, 3 = cloud shadow, ...). SCL is a **categorical** layer, so it must be reprojected with **nearest-neighbour** resampling (Week 5: there is no class 5.5).

In [ ]:
def bng_grid(bounds_bng, res=20):
    """An empty (NaN) DataArray template on a British National Grid grid."""
    xmin, ymin, xmax, ymax = bounds_bng
    x = np.arange(xmin + res / 2, xmax, res)
    y = np.arange(ymax - res / 2, ymin, -res)
    tpl = xr.DataArray(np.full((len(y), len(x)), np.nan, dtype='float32'),
                       coords={'y': y, 'x': x}, dims=('y', 'x'))
    return tpl.rio.write_crs(27700).rio.write_nodata(np.nan)


def mosaic_band(day_items, asset, template, bounds_ll, categorical=False, overview_level=None):
    """Read one band from several tiles, reproject each to the template grid and merge them."""
    merged = None
    for item in day_items:
        da = read_band(item, asset, bounds_ll, overview_level=overview_level)
        crs = da.rio.crs
        if categorical:
            da = da.where(da != 0).astype('float32')         # 0 = no data in SCL
        else:
            da = to_reflectance(da, item, asset)
        da = da.rio.write_crs(crs).rio.write_nodata(np.nan)
        resampling = Resampling.nearest if categorical else Resampling.bilinear
        on_grid = da.rio.reproject_match(template, resampling=resampling)
        merged = on_grid if merged is None else merged.combine_first(on_grid)
    return merged.rio.write_crs(27700)


if STAC_OK:
    try:
        bounds_bng = london_bng.total_bounds if BOUNDARIES_OK else \
            gpd.GeoSeries([box(*LONDON_BBOX)], crs=4326).to_crs(27700).total_bounds
        template = bng_grid(bounds_bng, res=20)

        red = mosaic_band(day_items, 'red', template, LONDON_BBOX, overview_level=0)   # 10 m band -> first overview = 20 m
        nir = mosaic_band(day_items, 'nir', template, LONDON_BBOX, overview_level=0)
        scl = mosaic_band(day_items, 'scl', template, LONDON_BBOX, categorical=True)  # native 20 m
        print('grid:', red.shape, '| CRS:', red.rio.crs, '| pixel size:', red.rio.resolution())
    except Exception as e:
        STAC_OK = False
        print(f'Reading the COGs failed ({type(e).__name__}: {e}). Section 3 will be skipped.')

In [ ]:
if STAC_OK:
    # Sanity check (section 3.3): reflectance should be between 0 and 1, and red over a city is low
    r = red.values[np.isfinite(red.values)]
    print(f'red reflectance: median {np.median(r):.3f}, 1st-99th percentile {np.percentile(r, 1):.3f}–{np.percentile(r, 99):.3f}, '
          f'negative: {100 * (r < 0).mean():.2f} %')

    CLEAR_CLASSES = [2, 4, 5, 6, 7, 11]          # dark area, vegetation, bare/built, water, unclassified, snow
    clear = scl.isin(CLEAR_CLASSES)
    print(f'cloud/shadow-masked share of the grid: {100 * (scl.notnull() & ~clear).sum().item() / scl.notnull().sum().item():.1f} %')

    ndvi = ((nir - red) / (nir + red)).where(clear)
    ndvi = ndvi.rio.write_crs(27700).rio.write_nodata(np.nan)

## 3.5 Clip to London and map

`rio.clip()` masks everything outside a polygon (both must be in the same CRS, here EPSG:27700). Because we told rioxarray that the no-data value is `NaN`, the area outside London stays empty rather than being filled with 0, which would show up as a false NDVI value.

In [ ]:
if STAC_OK and BOUNDARIES_OK:
    ndvi_london = ndvi.rio.clip(boroughs_bng.geometry, drop=True)

    fig, ax = plt.subplots(figsize=(11, 8))
    ndvi_london.plot.imshow(ax=ax, cmap='RdYlGn', vmin=-0.1, vmax=0.9,
                            cbar_kwargs={'label': 'NDVI (–)', 'shrink': 0.7})
    boroughs_bng.boundary.plot(ax=ax, color='black', linewidth=0.4)
    ax.set_title(f"Sentinel-2 NDVI, Greater London, {chosen['date']:%d %b %Y} "
                 f"(tiles {chosen['tiles']}, 20 m, clouds masked)")
    ax.set_xlabel('Easting (m, British National Grid)')
    ax.set_ylabel('Northing (m, British National Grid)')
    ax.set_aspect('equal')
    fig.savefig(FIG_DIR / 'london_ndvi_stac.png', dpi=200, bbox_inches='tight')
    plt.show()

Look for Richmond Park and Bushy Park (south-west), Epping Forest (north-east edge), Hampstead Heath, the Green Belt edge, the low NDVI of the dense centre, the Thames, the Docklands and Heathrow airport.

**Question:** A colleague's version of this map was dark green almost everywhere, even in the City of London. What are the two most likely causes?

<details><summary><b>Show answer</b></summary>

(1) **Applying the reflectance offset twice** (section 3.3): if red reflectance is pushed to about 0, NDVI = (NIR − red)/(NIR + red) approaches 1 everywhere. (2) **No-data filled with 0** instead of NaN, which mixes fake values into the image and its statistics. Both are silent errors: the code runs, but the numbers are wrong. Only a physical sanity check reveals them.

</details>

## 3.6 Zonal statistics: mean NDVI per borough

**Zonal statistics** summarise raster values inside each polygon. It is one of the most common operations in EO analysis, and it appears in the Part B urban heat island template. The function also reports what **fraction of each borough** had valid (cloud-free) pixels, so we know how much to trust each mean.

In [ ]:
def zonal_mean(raster, gdf, name_col):
    """Mean raster value and valid-pixel fraction for each polygon.

    Parameters
    ----------
    raster : xarray.DataArray
        2-D raster with a CRS (rioxarray), NaN = no data.
    gdf : geopandas.GeoDataFrame
        Polygons in the same, projected CRS as the raster.
    name_col : str
        Column with the polygon names.

    Returns
    -------
    pandas.DataFrame
        Columns: name_col, 'mean', 'valid_fraction'.
    """
    pixel_area = abs(raster.rio.resolution()[0] * raster.rio.resolution()[1])
    rows = []
    for _, row in gdf.iterrows():
        try:
            values = raster.rio.clip([row.geometry], drop=True).values
            n_valid = int(np.isfinite(values).sum())
        except Exception:                      # polygon entirely outside the raster
            values, n_valid = np.array([np.nan]), 0
        rows.append({name_col: row[name_col],
                     'mean': float(np.nanmean(values)) if n_valid else np.nan,
                     'valid_fraction': min(1.0, n_valid * pixel_area / row.geometry.area)})
    return pd.DataFrame(rows)


if STAC_OK and BOUNDARIES_OK:
    ndvi_by_borough = zonal_mean(ndvi, boroughs_bng, 'borough').sort_values('mean', ascending=False)
    print('Greenest:\n', ndvi_by_borough.head(3).round(3).to_string(index=False))
    print('\nLeast green:\n', ndvi_by_borough.tail(3).round(3).to_string(index=False))

    joined = boroughs_bng.merge(ndvi_by_borough, on='borough')
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    joined.plot(ax=axes[0], column='mean', cmap='YlGn', edgecolor='grey', linewidth=0.4, legend=True,
                legend_kwds={'label': 'Mean NDVI (–)', 'shrink': 0.7})
    axes[0].set_title(f"Mean NDVI by borough, {chosen['date']:%d %b %Y}")
    joined.plot(ax=axes[1], column='valid_fraction', cmap='Greys', vmin=0, vmax=1, edgecolor='grey',
                linewidth=0.4, legend=True, legend_kwds={'label': 'Share of borough with valid pixels', 'shrink': 0.7})
    axes[1].set_title('Data quality: valid (cloud-free) share')
    for ax in axes:
        ax.set_axis_off()
    fig.tight_layout()
    plt.show()

**Question:** Why is it worth mapping the valid fraction next to the result?

<details><summary><b>Show answer</b></summary>

A borough mean based on 40 % of its area (the rest under cloud) is less reliable than one based on 99 %, and it may be biased: clouds are not random (e.g. convective clouds may form over particular areas). Reporting data coverage is part of being honest about **uncertainty**, which is assessed in the Part B interpretation section. If coverage is poor, use another date or a multi-date composite (section 4).

</details>

---
# 4. Processing in the cloud: Google Earth Engine

**Google Earth Engine (GEE)** hosts **petabytes** of analysis-ready data (Landsat since 1972, Sentinel-1/2/3/5P, MODIS, ERA5, DEMs, land cover, forest change, population...) and runs your computations on **Google's servers**.

The key idea is **server-side** processing. Python objects such as `ee.Image` are only **descriptions** of a computation, and nothing is calculated until you ask for a result (a map tile, a number with `.getInfo()`, or an export). This lets you process years of imagery over a whole country in seconds, but it also means you cannot use NumPy on `ee` objects directly.

| Object | What it is |
|---|---|
| `ee.Image` | one raster (one or more bands) |
| `ee.ImageCollection` | a stack of images, e.g. all Sentinel-2 scenes |
| `ee.Geometry` | a point, line or polygon |
| `ee.Feature` / `ee.FeatureCollection` | geometry + attributes, i.e. the GEE version of a GeoDataFrame |
| `ee.Reducer` | a summary: mean, median, percentile, count... |

## 4.1 Set up Earth Engine (one-off, about 5 minutes)

1. Go to <https://code.earthengine.google.com/register> with your Google account and choose **unpaid, noncommercial use** (academic / student). Earth Engine is free for this.
2. **Create a Google Cloud project** when asked, and note its **project ID** (e.g. `ee-12345678` or `geog0111-12345678`).
3. **Part B anonymity rule:** name your project with your **student number** or a neutral name, **not your real name**, because the project ID appears in your submitted notebook.
4. Put the project ID in the cell below and run it. The first time, a pop-up asks you to authorise access; accept it.

In [ ]:
import ee
import geemap

PROJECT_ID = 'YOUR-PROJECT-ID'      # <-- replace with your own Earth Engine project ID

def init_gee(project_id):
    # Authenticate (first time only) and initialise Earth Engine. Return True on success.
    if project_id in (None, '', 'YOUR-PROJECT-ID'):
        print('Set PROJECT_ID first (section 4.1). Earth Engine cells will be skipped.')
        return False
    try:
        ee.Initialize(project=project_id)
    except Exception:
        try:
            ee.Authenticate()
            ee.Initialize(project=project_id)
        except Exception as e:
            print(f'Could not initialise Earth Engine: {e}')
            return False
    print('Earth Engine initialised with project', project_id)
    return True

GEE_AVAILABLE = init_gee(PROJECT_ID)

try:                                    # interactive maps in Colab
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

## 4.2 Bring our boundaries into Earth Engine

`geemap.gdf_to_ee()` converts a GeoDataFrame (in EPSG:4326) into an `ee.FeatureCollection`, so the polygons we prepared in Python can be used on Google's servers.

In [ ]:
if GEE_AVAILABLE:
    if BOUNDARIES_OK:
        boroughs_ee = geemap.gdf_to_ee(boroughs[['borough', 'geometry']])
        london_ee = boroughs_ee.geometry().dissolve()     # one Greater London geometry
    else:
        london_ee = ee.Geometry.Rectangle(list(LONDON_BBOX))
        boroughs_ee = ee.FeatureCollection([ee.Feature(london_ee, {'borough': 'London (bbox)'})])
    borough_outlines = boroughs_ee.style(color='black', fillColor='00000000', width=1)
    print('Number of boroughs in Earth Engine:', boroughs_ee.size().getInfo())

## 4.3 A cloud-free Sentinel-2 composite and NDVI

A single date is often partly cloudy (as section 3 showed). Instead, we take **all** Sentinel-2 L2A images of the summer, **mask cloudy pixels** in each one, and take the **median** of what is left for every pixel. The result is a **composite**: a clean image for the whole season.

For cloud masking we use **Cloud Score+**, a Google product that gives every Sentinel-2 pixel a clear-sky score between 0 and 1 (band `cs_cdf`). We keep pixels with a score ≥ 0.6.

Note the Earth Engine band names: `B4` (red) and `B8` (NIR). They are **not** the same as the STAC asset names (`red`, `nir`). And this `HARMONIZED` collection, like Earth Search, has already removed the offset from newer scenes, so the scaling is just × 0.0001. **Each data source has its own conventions, and its catalogue page tells you which.**

In [ ]:
def s2_composite(region, start, end, clear_threshold=0.6):
    """Median Sentinel-2 L2A composite with Cloud Score+ masking, scaled to reflectance.

    Parameters
    ----------
    region : ee.Geometry
        Area of interest.
    start, end : str
        Date range 'YYYY-MM-DD' (end exclusive).
    clear_threshold : float, optional
        Minimum Cloud Score+ cs_cdf value to keep a pixel (default 0.6).

    Returns
    -------
    ee.Image
        Median composite of the optical bands, as reflectance (0-1).
    """
    cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(region)
          .filterDate(start, end)
          .linkCollection(cs_plus, ['cs_cdf']))
    masked = s2.map(lambda img: img.updateMask(img.select('cs_cdf').gte(clear_threshold)))
    return (masked.select(['B2', 'B3', 'B4', 'B8', 'B11'])
                  .median()
                  .multiply(0.0001)              # stored DN -> reflectance
                  .clip(region))


if GEE_AVAILABLE:
    s2_2024 = s2_composite(london_ee, '2024-06-01', '2024-09-01')
    ndvi_2024 = s2_2024.normalizedDifference(['B8', 'B4']).rename('NDVI')

    rgb_vis = {'bands': ['B4', 'B3', 'B2'], 'min': 0.0, 'max': 0.25, 'gamma': 1.2}
    ndvi_vis = {'min': -0.1, 'max': 0.8, 'palette': ['#a50026', '#f46d43', '#fee08b', '#d9ef8b', '#66bd63', '#006837']}

    Map = geemap.Map()
    Map.centerObject(london_ee, 10)
    Map.addLayer(s2_2024, rgb_vis, 'Sentinel-2 true colour, summer 2024')
    Map.addLayer(ndvi_2024, ndvi_vis, 'NDVI, summer 2024')
    Map.addLayer(borough_outlines, {}, 'Boroughs')
    Map.add_colorbar(ndvi_vis, label='NDVI (–)', layer_name='NDVI, summer 2024')
    Map.addLayerControl()
    display(Map)

Use the layer control (top right) to switch layers, and zoom in to Regent's Park, the Olympic Park or Heathrow.

## 4.4 Numbers from the server: `reduceRegion` and `reduceRegions`

Maps are for looking; for analysis we need **numbers**. A **reducer** summarises the pixels inside a region, and `reduceRegions` does it for every feature at once: zonal statistics on the server. `scale` is the pixel size (in metres) at which the reducer works.

In [ ]:
if GEE_AVAILABLE:
    mean_ndvi = ndvi_2024.reduceRegion(reducer=ee.Reducer.mean(), geometry=london_ee,
                                       scale=20, maxPixels=1e9).getInfo()
    print('Mean NDVI of Greater London, summer 2024:', round(mean_ndvi['NDVI'], 3))

    ndvi_boroughs_ee = ndvi_2024.reduceRegions(collection=boroughs_ee, reducer=ee.Reducer.mean(), scale=20)
    ndvi_gee = geemap.ee_to_df(ndvi_boroughs_ee)[['borough', 'mean']].rename(columns={'mean': 'ndvi'})
    display(ndvi_gee.sort_values('ndvi', ascending=False).head())

**Question:** The borough means from Earth Engine (summer composite) and from section 3 (one date) are similar but not identical. Why?

<details><summary><b>Show answer</b></summary>

They are different inputs: a median of all cloud-free observations over three months versus a single day; different cloud masks (Cloud Score+ vs SCL); different resampling (20 m overview vs Earth Engine's own resampling at `scale=20`). Vegetation also changes through the summer, e.g. grass browning in a dry July. Neither is "the" NDVI. Report which method you used, and why.

</details>

## 4.5 Land surface temperature from Landsat

The **urban heat island** topic needs temperature. Landsat 8 and 9 have a **thermal** band, and the Collection 2 Level-2 product provides **land surface temperature (LST)** in band `ST_B10`: the temperature of the surface itself (roofs, roads, grass), not of the air. It is stored as integers, and the catalogue page gives the conversion to kelvin:

$$T_{K} = DN \times 0.00341802 + 149.0$$

Clouds are removed using the `QA_PIXEL` band, where individual **bits** flag dilated cloud (bit 1), cloud (bit 3) and cloud shadow (bit 4).

In [ ]:
def landsat_lst(img):
    """Cloud-masked land surface temperature (°C) from a Landsat 8/9 C2 L2 image."""
    qa = img.select('QA_PIXEL')
    clear = (qa.bitwiseAnd(1 << 1).eq(0)          # dilated cloud
               .And(qa.bitwiseAnd(1 << 3).eq(0))   # cloud
               .And(qa.bitwiseAnd(1 << 4).eq(0)))  # cloud shadow
    lst = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
    return lst.updateMask(clear).copyProperties(img, ['system:time_start'])


def summer_lst(region, start, end, max_cloud=30):
    """Median LST (°C) from Landsat 8 and 9 over a region and date range."""
    landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
               .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
               .filterBounds(region)
               .filterDate(start, end)
               .filter(ee.Filter.lt('CLOUD_COVER', max_cloud)))
    print('Landsat scenes used:', landsat.size().getInfo())
    return landsat.map(landsat_lst).median().clip(region)


if GEE_AVAILABLE:
    lst_2024 = summer_lst(london_ee, '2024-06-01', '2024-09-01')
    lst_vis = {'min': 20, 'max': 40, 'palette': ['#313695', '#74add1', '#ffffbf', '#f46d43', '#a50026']}

    Map = geemap.Map()
    Map.centerObject(london_ee, 10)
    Map.addLayer(lst_2024, lst_vis, 'Landsat LST, summer 2024 (°C)')
    Map.addLayer(borough_outlines, {}, 'Boroughs')
    Map.add_colorbar(lst_vis, label='Land surface temperature (°C)', layer_name='Landsat LST, summer 2024 (°C)')
    display(Map)

## 4.6 Is greener cooler?

Now we combine the two layers: mean LST per borough (with `scale=30`, as Landsat thermal data are delivered at 30 m), merged with mean NDVI from section 4.4, and plotted against each other.

In [ ]:
if GEE_AVAILABLE:
    lst_boroughs = lst_2024.reduceRegions(collection=boroughs_ee, reducer=ee.Reducer.mean(), scale=30)
    lst_df = geemap.ee_to_df(lst_boroughs)[['borough', 'mean']].rename(columns={'mean': 'lst'})
    green_heat = ndvi_gee.merge(lst_df, on='borough')
    green_heat.to_csv(DATA_DIR / 'processed' / 'london_boroughs_ndvi_lst_2024.csv', index=False)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(green_heat['ndvi'], green_heat['lst'], color='tab:red')
    for _, r in green_heat.iterrows():
        if r['borough'] in ['City of London', 'Westminster', 'Tower Hamlets', 'Newham',
                            'Richmond upon Thames', 'Bromley', 'Havering', 'Hillingdon']:
            ax.annotate(r['borough'], (r['ndvi'], r['lst']), fontsize=8, xytext=(3, 3), textcoords='offset points')
    ax.set_xlabel('Mean summer NDVI (–)')
    ax.set_ylabel('Mean summer land surface temperature (°C)')
    ax.set_title('London boroughs, summer 2024: vegetation vs surface temperature')
    fig.savefig(FIG_DIR / 'london_ndvi_vs_lst.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Pearson correlation:', round(green_heat['ndvi'].corr(green_heat['lst']), 2))

**Question:** What relationship do you see, what mechanisms could explain it, and what could explain boroughs that don't follow the trend?

<details><summary><b>Show answer</b></summary>

Greener boroughs have cooler surfaces (a strong negative correlation): vegetation shades the ground and cools it through evapotranspiration, while roofs, roads and paving absorb and store heat. Outliers can come from **large water bodies** (reservoirs and the Thames are cool but have low NDVI), **airports and industrial land** (Heathrow in Hillingdon), **dry grass** (low NDVI but not necessarily hot in the same way), or a different number of clear Landsat scenes per borough. Note also that this is a **correlation across 33 areas**: it does not prove causation, and borough averages hide large differences within each borough.

</details>

## 4.7 Change over time

The urban heat island topic compares a **baseline** and a **recent** year. Thanks to `s2_composite()`, a 2018 composite is one line of code. Brown means NDVI decreased, and blue-green means it increased:

In [ ]:
if GEE_AVAILABLE:
    ndvi_2018 = s2_composite(london_ee, '2018-06-01', '2018-09-01').normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndvi_diff = ndvi_2024.subtract(ndvi_2018).rename('dNDVI')
    diff_vis = {'min': -0.3, 'max': 0.3, 'palette': ['#8c510a', '#d8b365', '#f5f5f5', '#5ab4ac', '#01665e']}

    Map = geemap.Map()
    Map.centerObject(london_ee, 10)
    Map.addLayer(ndvi_diff, diff_vis, 'NDVI change, 2024 minus 2018')
    Map.addLayer(borough_outlines, {}, 'Boroughs')
    Map.add_colorbar(diff_vis, label='NDVI change (–)', layer_name='NDVI change, 2024 minus 2018')
    display(Map)

**Question:** Large parts of London's parks show a strong NDVI *increase* from 2018 to 2024. Has London become much greener?

<details><summary><b>Show answer</b></summary>

Not necessarily. Summer **2018 had an exceptional drought and heatwave**: lawns and parks turned brown, so NDVI was unusually low. The positive difference over parks is mostly **weather**, not land-use change. Genuine land-use change shows up as sharp, local patches (e.g. new developments in east London, with an NDVI decrease). A fair comparison uses similar seasons, averages over several years, or adds land-cover data, and states this as a limitation. This kind of reasoning is exactly what the Part B interpretation section rewards.

</details>

## 4.8 Exporting results to use in Python

Earth Engine is excellent at processing, but for final figures, statistics or other tools you often **export** a result:

- **small** exports (up to a few tens of MB): `geemap.ee_export_image()` downloads a GeoTIFF straight into Colab;
- **large** exports: `ee.batch.Export.image.toDrive()` runs as a background **task** and saves to your Google Drive (check progress in the *Tasks* tab of the [Code Editor](https://code.earthengine.google.com/)).

Choose `scale` (pixel size) and `crs` **deliberately**: a 10 m export of all of London is about 16 million pixels per band, while at 100 m it is 160 thousand.

In [ ]:
if GEE_AVAILABLE:
    export_tif = DATA_DIR / 'processed' / 'london_lst_2024_100m.tif'
    if not export_tif.exists():
        geemap.ee_export_image(lst_2024, filename=str(export_tif), scale=100,
                               region=london_ee, crs='EPSG:27700', file_per_band=False)

    if export_tif.exists():
        lst_local = rxr.open_rasterio(export_tif, masked=True).squeeze()
        print(lst_local.rio.crs, lst_local.shape, lst_local.rio.resolution())

        fig, ax = plt.subplots(figsize=(10, 7))
        lst_local.plot.imshow(ax=ax, cmap='RdYlBu_r', vmin=20, vmax=40,
                              cbar_kwargs={'label': 'Land surface temperature (°C)', 'shrink': 0.7})
        if BOUNDARIES_OK:
            boroughs_bng.boundary.plot(ax=ax, color='black', linewidth=0.3)
        ax.set_title('Landsat 8/9 median LST, summer 2024 (exported at 100 m)')
        ax.set_xlabel('Easting (m, British National Grid)')
        ax.set_ylabel('Northing (m, British National Grid)')
        ax.set_aspect('equal')
        fig.savefig(FIG_DIR / 'london_lst_2024.png', dpi=200, bbox_inches='tight')
        plt.show()

For a large export to Google Drive (not run automatically, because it writes to your Drive):

```python
task = ee.batch.Export.image.toDrive(
    image=ndvi_2024, description='london_ndvi_2024_10m', folder='GEE_exports',
    fileNamePrefix='london_ndvi_2024_10m', region=london_ee, scale=10,
    crs='EPSG:27700', maxPixels=1e10)
task.start()
print(task.status())
```

## 4.9 The same tools, another continent: forest loss in the Amazon

The second suggested Part B topic looks at **deforestation and fire** in the Amazon. Earth Engine hosts the **Hansen Global Forest Change** dataset (University of Maryland): tree cover in 2000 and annual tree-cover loss at 30 m since 2001. Here it is in **Rondônia**, the region of the satellite image you displayed in Week 5. Switch the base map to satellite imagery in the layer control to compare.

In [ ]:
if GEE_AVAILABLE:
    gfc = ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
    rondonia = ee.Geometry.Rectangle([-63.5, -10.5, -62.0, -9.5])     # about 165 x 110 km

    tree2000 = gfc.select('treecover2000')
    tree2000 = tree2000.updateMask(tree2000.gt(0)).clip(rondonia)                # hide 0 % so the base map shows
    loss_year = gfc.select('lossyear')
    loss_year = loss_year.updateMask(loss_year.gt(0)).add(2000).clip(rondonia)   # 1..25 -> 2001..2025

    tree_vis = {'min': 0, 'max': 100, 'palette': ['#edf8e9', '#74c476', '#00441b']}
    loss_vis = {'min': 2001, 'max': 2025, 'palette': ['#ffffcc', '#fd8d3c', '#e31a1c', '#800026']}

    Map = geemap.Map()
    Map.centerObject(rondonia, 9)
    Map.addLayer(tree2000, tree_vis, 'Tree cover in 2000 (%)')
    Map.addLayer(loss_year, loss_vis, 'Year of forest loss')
    Map.add_colorbar(tree_vis, label='Tree cover in 2000 (%)', layer_name='Tree cover in 2000 (%)')
    Map.add_colorbar(loss_vis, label='Year of forest loss', layer_name='Year of forest loss')
    Map.addLayerControl()
    display(Map)

How much forest was lost each year? A **grouped reducer** sums the area of loss pixels (in km²), grouped by year, over the ~20 million 30 m pixels in the rectangle, and returns just 23 numbers. That is the power of server-side processing.

In [ ]:
if GEE_AVAILABLE:
    forest2000 = gfc.select('treecover2000').gte(30)                  # 'forest' = at least 30 % tree cover in 2000
    loss_km2 = gfc.select('loss').And(forest2000).multiply(ee.Image.pixelArea()).divide(1e6)
    stats = (loss_km2.addBands(gfc.select('lossyear'))
             .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName='year'),
                           geometry=rondonia, scale=30, maxPixels=1e10)
             .getInfo())
    loss_df = pd.DataFrame(stats['groups']).rename(columns={'sum': 'loss_km2'})
    loss_df['year'] = loss_df['year'] + 2000
    loss_df = loss_df[loss_df['year'] > 2000]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(loss_df['year'], loss_df['loss_km2'], color='#e31a1c')
    ax.set_xlabel('Year')
    ax.set_ylabel('Forest loss (km²)')
    ax.set_title('Annual tree-cover loss in a Rondônia study rectangle (Hansen GFC v1.13)')
    plt.show()

**Question:** Hansen "loss" is not the same as deforestation. Why does the difference matter for the Amazon topic?

<details><summary><b>Show answer</b></summary>

Hansen loss is any **stand-replacement disturbance**, i.e. the loss of tree canopy in a 30 m pixel. It includes clearing for pasture (deforestation), but also **fire**, wind damage, river dynamics and harvesting in plantations, and it does not record regrowth in the same way. To separate fire from clearing, combine it with a burned-area product (MODIS MCD64A1, as in the Part B template). The definition of "forest" (here ≥ 30 % tree cover in 2000) also changes the numbers, which is another choice to state and justify.

</details>

---
# 5. Finding the right dataset

Start with the **question**, not the code: what variable, at what spatial and temporal resolution, for which period and area? Then search the catalogues and **read the dataset page** (bands, scale factors, units, dates, licence, citation) **before** you write code.

- **Earth Engine Data Catalog** (official): <https://developers.google.com/earth-engine/datasets>
- **Awesome GEE Community Catalog** (research datasets shared by the community, e.g. canopy height, biomass, urban products): <https://gee-community-catalog.org/>. Always check the original source, version and licence.
- **STAC catalogues:** Earth Search <https://earth-search.aws.element84.com/v1>, Microsoft Planetary Computer <https://planetarycomputer.microsoft.com/catalog>; browse visually with <https://radiantearth.github.io/stac-browser/>.
- **Quick-look browsers** to explore imagery without code: Copernicus Browser <https://browser.dataspace.copernicus.eu/>, NASA Worldview <https://worldview.earthdata.nasa.gov/>.

| Variable | Typical datasets (Earth Engine ID or source) |
|---|---|
| Surface reflectance, NDVI | Sentinel-2 `COPERNICUS/S2_SR_HARMONIZED` (10 m, 2017–); Landsat `LANDSAT/LC09/C02/T1_L2` (30 m) |
| Land surface temperature | Landsat C2 L2 `ST_B10` (30 m delivered); MODIS `MODIS/061/MOD11A2` (1 km, 8-day) |
| Air temperature, precipitation | ERA5-Land `ECMWF/ERA5_LAND/MONTHLY_AGGR` (~9 km); Met Office station data (Week 4) |
| Forest loss | Hansen `UMD/hansen/global_forest_change_2025_v1_13` (30 m) |
| Burned area | MODIS `MODIS/061/MCD64A1` (500 m, monthly) |
| Land cover | ESA WorldCover `ESA/WorldCover/v200` (10 m, 2021); Dynamic World `GOOGLE/DYNAMICWORLD/V1` (10 m, near real-time) |
| Elevation | Copernicus DEM `COPERNICUS/DEM/GLO30` (30 m) |

For example, ESA WorldCover over London (it comes with its own class colours):

In [ ]:
if GEE_AVAILABLE:
    worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(london_ee)
    Map = geemap.Map()
    Map.centerObject(london_ee, 10)
    Map.addLayer(worldcover, {}, 'ESA WorldCover 2021')
    Map.add_legend(builtin_legend='ESA_WorldCover')
    display(Map)

---
# 6. Where we are heading: Part B, your EO project

From now on, every session builds towards **Part B**: a self-contained data-analysis project in **one notebook**, submitted in the **first week of Term 2**.

## What it must include

- **EO data** (e.g. Sentinel-2, Landsat, MODIS);
- data with a **temporal** component (a change between dates, or a time series);
- a **series of your own functions**, with clear arguments, return values and docstrings, that do not rely on notebook variables (Week 3).

Both suggested topics also combine a **text/CSV** dataset and a **vector** dataset with the EO imagery.

## How it is marked

| Section | Weight | What we look for |
|---|---|---|
| **Introduction** | 15 % | A focused question; short background; data description (source, type, resolution, period); a plan |
| **Data processing** | **60 %** | Clear, reproducible, **function-based** code; loading and checks; appropriate processing (masking, indices, aggregation); saved outputs |
| **Interpretation** | 20 % | Clear static figures and tables with captions; what the results show; **limitations and uncertainty** |
| **Conclusion** | 5 % | What you did and found; answer to the question; 1–3 ideas for improvement |

**Rules:** submit as PDF (Restart → Run all → export) with the `.ipynb` as backup; **no names**, only your student number, including in your GEE project ID; AI may help with **code without comments** only, and all comments, text and interpretation must be your own; include an AI-use declaration.

## Two suggested topics, or your own

| | Topic 1: Urban heat island change in Greater London | Topic 2: Deforestation and fire–climate interactions in the Amazon |
|---|---|---|
| **Question** | How have vegetation, built-up surfaces and surface temperature changed between two years, and how do they relate? | How does ENSO (El Niño vs a baseline year) relate to climate, deforestation and fire? |
| **Text / CSV data** | Met Office station record, e.g. Heathrow (**Weeks 4–5**) | NOAA Oceanic Niño Index (**Weeks 4–5**) |
| **Vector data** | Borough boundaries dissolved to Greater London (**today, section 2**) | Amazon Basin boundary |
| **EO data** | Sentinel-2 NDVI/NDBI and Landsat LST (**today, sections 4.3–4.8**) | ERA5-Land climate, Hansen forest loss (**today, 4.9**), MODIS burned area |
| **Typical figures** | NDVI, NDBI and LST maps for two years; difference maps; zonal statistics by borough; station time series | ONI time series with years marked (**Week 5**); loss and burned-area maps; summary table |

You can adapt either topic (a different **city**, region, years or index) or propose **your own**, provided it meets the requirements. Templates for both topics and the full Part B guidance are on Moodle and Colab: <https://colab.research.google.com/drive/1X2N6Qg80QG3aFLeG3rCFSQcZhgrEfyjX?usp=sharing>

> **Look at what you have already built:** `read_met_station()` (Week 4), plotting functions and ONI plots (Week 5), and `get_geoboundaries()`, `mosaic_band()`, `zonal_mean()`, `s2_composite()` and `summer_lst()` (today). These are the components of a Part B notebook.

### Think about your project before the next session

Jot down a first idea (you can change it later). We will discuss them together.

- **Working title** and **question** (one sentence);
- **study area**, and where its boundary comes from;
- **years** compared, or the time period;
- **datasets**: text/CSV · vector · EO (with resolution);
- which of today's functions you can reuse, and which new ones you will need;
- **one risk** (clouds, data gaps, file size...) and how you would handle it.

---
# Summary

| Topic | Key points |
|---|---|
| Access modes | download files · read cloud-optimised files found via STAC · process server-side in Earth Engine |
| Vector | `gpd.read_file`, `.crs`, `.to_crs(27700)` before areas, `.dissolve()`, `.to_file(... .gpkg)`, choropleths |
| STAC + COG | `Client.open(...).search(collections, bbox, datetime, query)`; items and assets; `rxr.open_rasterio(href, overview_level=...)` plus `rio.clip_box` reads only what you need |
| Scaling | reflectance = DN × scale + offset; read the metadata **and** sanity-check the values; set nodata to NaN |
| Mosaics and masks | reproject tiles to one grid (`reproject_match`), `combine_first`; categorical layers (SCL) need nearest-neighbour |
| Clip and zonal stats | `rio.clip`; mean per polygon plus valid fraction |
| Earth Engine | `ee.Initialize(project=...)`; collection → filter → map(mask) → `median()`; reducers; `getInfo()` only for small results; export with `geemap.ee_export_image` or `Export.image.toDrive` |
| Datasets | check IDs, band names, scale factors and licences on the catalogue page; never trust AI-invented IDs |

# Resources

| Resource | What it is |
|---|---|
| [GeoPandas user guide](https://geopandas.org/en/stable/docs/user_guide.html) | Reading, projecting, plotting, spatial joins, overlays |
| [rioxarray documentation](https://corteva.github.io/rioxarray/stable/) | Opening, clipping and reprojecting rasters with xarray |
| [STAC introduction](https://stacspec.org/en/tutorials/intro-to-stac/) | What STAC catalogues, items and assets are |
| [Earth Engine Python guides](https://developers.google.com/earth-engine/guides) | Concepts: images, collections, reducers, exports |
| [geemap examples](https://geemap.org/notebooks/) | Hundreds of notebook examples for Earth Engine in Python |
| [Cloud-Based Remote Sensing with Google Earth Engine (EEFA book)](https://www.eefabook.org/) | Free open textbook with worked examples, from beginner to advanced |
| [Earth Data Science textbook (Earth Lab)](https://www.earthdatascience.org/courses/use-data-open-source-python/) | Open-source Python for vector and raster data |
| [Copernicus Browser](https://browser.dataspace.copernicus.eu/) | Look at any Sentinel scene in the browser, with no code |